# Validation Report: Cross-check Numbers and Coherence

This notebook validates that numbers reported in LaTeX documents match the actual data and code.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from cannabis_tax.analysis import build_propensity_dataset, prepare_propensity_regression_data

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

## 1. Load the data and check sample sizes

In [ ]:
# Build the dataset
df = build_propensity_dataset()
print(f"Total rows in propensity dataset: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumns: {list(df.columns)}")

## 2. Check the discrepancy: 3982 vs. 3980

**Question**: 
- `variables.tex` claims 3982 observations in source data
- `paper.tex` reports 3980 in MCO regression
- Where did 2 observations go?

In [ ]:
# Check how many observations have all required variables for MCO full model
cols_mco_full = ['propension_12m', 'edad', 'edad_cuadrado', 'sexo_label', 'educacion_grupo']
mco_full_complete = df[cols_mco_full].dropna()
print(f"Rows with complete data for MCO full model: {len(mco_full_complete)}")

# Check per variable
print("\nMissingness by variable (MCO full model required):")
for col in cols_mco_full:
    missing = df[col].isna().sum()
    print(f"  {col}: {missing} missing ({100*missing/len(df):.2f}%)")

## 3. Check price availability breakdown

**Key numbers from `variables.tex`**:
- Total observations with price: 823
- Among non-consumers: 220 with price
- Among consumers: 603 with price

Let's validate:

In [ ]:
# Check price availability
print("Price availability:")
print(f"Total observations with price: {df['precio_compra'].notna().sum()}")
print(f"Total observations without price: {df['precio_compra'].isna().sum()}")

# Among consumers
consumers = df[df['propension_12m'] == 1]
consumers_with_price = consumers['precio_compra'].notna().sum()
print(f"\nAmong consumers (propension_12m=1): {len(consumers)} total")
print(f"  With price: {consumers_with_price}")
print(f"  Without price: {consumers['precio_compra'].isna().sum()}")

# Among non-consumers
non_consumers = df[df['propension_12m'] == 0]
non_consumers_with_price = non_consumers['precio_compra'].notna().sum()
print(f"\nAmong non-consumers (propension_12m=0): {len(non_consumers)} total")
print(f"  With price: {non_consumers_with_price}")
print(f"  Without price: {non_consumers['precio_compra'].isna().sum()}")

# Summary
total_with_price = consumers_with_price + non_consumers_with_price
print(f"\nTotal with price (sum): {total_with_price}")
print(f"Does it match variables.tex report (823)? {total_with_price == 823}")

## 4. Check MCO with price sample size

**Paper claims**: 755 observations for MCO with price model

In [ ]:
# MCO with price
cols_mco_price = cols_mco_full + ['log_precio_compra']
mco_price_complete = df[cols_mco_price].dropna()
print(f"Rows with complete data for MCO price model: {len(mco_price_complete)}")
print(f"Does it match paper (755)? {len(mco_price_complete) == 755}")

# Where are we losing data?
print(f"\nData loss breakdown:")
print(f"  Total rows: {len(df)}")
print(f"  After requiring education: {df['educacion_grupo'].notna().sum()}")
print(f"  After requiring price: {df[cols_mco_price].notna().all(axis=1).sum()}")

## 5. Validate education distribution

**Question**: Are there stable counts in each education group?

In [ ]:
print("Education group distribution:")
print(df['educacion_grupo'].value_counts(dropna=False))

print("\nEducation group in MCO full model:")
print(mco_full_complete['educacion_grupo'].value_counts())

## 6. Check the assumption: missing = non-consumer?

**Key question**: The code treats missing `consumo_12m` as 0 (non-consumer). Is this correct?

In [ ]:
print("Original consumo_12m value counts:")
print(df['consumo_12m'].value_counts(dropna=False))

print("\nAfter recoding to propension_12m (binary):")
print(df['propension_12m'].value_counts(dropna=False))

print(f"\nMarking: consumo_12m missing = {df[df['consumo_12m_missing_original']==1].shape[0]} cases")
print(f"These are treated as propension_12m = 0 (non-consumer)")

## 7. Create a cross-check table

Summary of all reported vs. actual numbers

In [ ]:
validation_data = {
    'Metric': [
        'Source observations (variables.tex)',
        'Actual rows in dataset',
        'MCO full model (paper.tex)',
        'Actual MCO full complete cases',
        'MCO price model (paper.tex)',
        'Actual MCO price complete cases',
        'Price: total with data',
        'Price: among consumers',
        'Price: among non-consumers',
    ],
    'Reported': [3982, np.nan, 3980, np.nan, 755, np.nan, 823, 603, 220],
    'Actual': [
        len(df),
        len(df),
        len(mco_full_complete),
        len(mco_full_complete),
        len(mco_price_complete),
        len(mco_price_complete),
        df['precio_compra'].notna().sum(),
        consumers_with_price,
        non_consumers_with_price,
    ]
}

validation_df = pd.DataFrame(validation_data)
validation_df['Match'] = validation_df.apply(
    lambda row: 'N/A' if pd.isna(row['Reported']) else ('✓' if row['Reported'] == row['Actual'] else '✗'),
    axis=1
)

print(validation_df.to_string(index=False))

## 8. Coefficient stability check

Do we expect the coefficients to be comparable between full and price models?

In [ ]:
# Compare sample composition
print("Sample composition check:")
print(f"\nFull model (n={len(mco_full_complete)}):")
print(f"  Consumers: {(mco_full_complete['propension_12m']==1).sum()} ({100*(mco_full_complete['propension_12m']==1).mean():.1f}%)")
print(f"  Non-consumers: {(mco_full_complete['propension_12m']==0).sum()} ({100*(mco_full_complete['propension_12m']==0).mean():.1f}%)")
print(f"  Mean age: {mco_full_complete['edad'].mean():.1f}")
print(f"  Female: {(mco_full_complete['sexo_label']=='Mujer').mean()*100:.1f}%")

print(f"\nPrice model (n={len(mco_price_complete)}):")
print(f"  Consumers: {(mco_price_complete['propension_12m']==1).sum()} ({100*(mco_price_complete['propension_12m']==1).mean():.1f}%)")
print(f"  Non-consumers: {(mco_price_complete['propension_12m']==0).sum()} ({100*(mco_price_complete['propension_12m']==0).mean():.1f}%)")
print(f"  Mean age: {mco_price_complete['edad'].mean():.1f}")
print(f"  Female: {(mco_price_complete['sexo_label']=='Mujer').mean()*100:.1f}%")

print("\n⚠️  INTERPRETATION:")
print("If percentages differ much, this explains why coefficients change between models.")

## Summary of Validation

### Key Findings:

1. **Sample size discrepancy**: Source data claims 3982, actual is ___. Where did the gap come from?

2. **MCO models**: Full model has ___ obs, price model has ___.

3. **Price availability**: ___ total with price. Distribution check needed.

4. **Selection bias**: Sample composition differs between full and price models, which explains coefficient changes.

### Recommendations:

- [ ] Explain the 3982 vs 3980 discrepancy in the paper
- [ ] Include price availability numbers in the main text (not just appendix)
- [ ] Consider testing for selection bias (e.g., Heckman procedure)
- [ ] Document the "missing = 0" assumption explicitly